<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/16-dates.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 16 — Dates, Times and Time Zones

Companion to [the chapter](https://www.ai.biz/books/python-primer/dates-and-times/).


In [ ]:
import pandas as pd, numpy as np


## 1. The ambiguity in every date


In [ ]:
print('03/04/2026 parsed as default  :', pd.to_datetime('03/04/2026').strftime('%d %B %Y'))
print('03/04/2026 with dayfirst=True :', pd.to_datetime('03/04/2026', dayfirst=True).strftime('%d %B %Y'))
print()
print('Same string, two different days. Always pass format when you know it.')


## 2. errors='coerce' shows you every failure at once


In [ ]:
raw = pd.Series(['2026-01-15','2026-02-30','not a date','2026-03-01','0000-00-00'])
parsed = pd.to_datetime(raw, format='%Y-%m-%d', errors='coerce')
bad = raw[parsed.isna()]
print(f'{len(bad)} unparseable:'); print(bad.tolist())
print(); print(parsed)


## 3. Extracting parts with .dt


In [ ]:
d = pd.Series(pd.to_datetime(['2026-01-15 09:30','2026-06-20 22:15','2026-12-25 00:05']))
print(pd.DataFrame({
    'year': d.dt.year, 'month': d.dt.month,
    'weekday': d.dt.day_name(), 'hour': d.dt.hour,
    'quarter': d.dt.quarter, 'is_weekend': d.dt.dayofweek >= 5,
}))


## 4. Cyclical encoding

Hour 23 and hour 0 are one hour apart. As raw numbers they are 23 apart.


In [ ]:
h = pd.Series([0, 6, 12, 18, 23])
enc = pd.DataFrame({'hour': h,
                    'sin': np.sin(2*np.pi*h/24).round(3),
                    'cos': np.cos(2*np.pi*h/24).round(3)})
print(enc)

def dist(a, b):
    ra, rb = enc[enc.hour==a].iloc[0], enc[enc.hour==b].iloc[0]
    return np.hypot(ra.sin-rb.sin, ra.cos-rb.cos)

print()
print(f'raw distance 23 to 0     : {abs(23-0)}')
print(f'encoded distance 23 to 0 : {dist(23,0):.3f}')
print(f'encoded distance 0 to 12 : {dist(0,12):.3f}  <- correctly far apart')


## 5. Timedelta vs DateOffset


In [ ]:
jan31 = pd.Timestamp('2026-01-31')
print('start                    :', jan31.date())
print('+ Timedelta(days=30)     :', (jan31 + pd.Timedelta(days=30)).date())
print('+ DateOffset(months=1)   :', (jan31 + pd.DateOffset(months=1)).date(), ' <- calendar aware')


## 6. tz_localize declares a zone; tz_convert changes one


In [ ]:
naive = pd.Series(pd.to_datetime(['2026-03-15 14:30']))
print('naive          :', naive.iloc[0], ' <- 14:30 where?')

london = naive.dt.tz_localize('Europe/London')
print('tz_localize    :', london.iloc[0])
print('tz_convert UTC :', london.dt.tz_convert('UTC').iloc[0])
print('tz_convert JST :', london.dt.tz_convert('Asia/Tokyo').iloc[0])

aware = pd.Series(pd.to_datetime(['2026-03-15 14:30'], utc=True))
try:
    naive.iloc[0] < aware.iloc[0]
except TypeError as e:
    print(); print('comparing naive with aware:', e)


That error is the library protecting you.


## 7. Resampling and rolling windows


In [ ]:
idx = pd.date_range('2026-01-01', periods=90, freq='D')
rng = np.random.default_rng(1)
ts = pd.Series(rng.normal(100, 20, 90).cumsum(), index=idx, name='revenue')

print('weekly sums:'); print(ts.resample('W').sum().head())
print()
print('7-day rolling mean, first values:')
print(ts.rolling(7).mean().head(8))
print()
print('with min_periods=1 there are no leading NaNs:')
print(ts.rolling(7, min_periods=1).mean().head(3))


## 8. shift(1) is a feature; shift(-1) is leakage


In [ ]:
f = pd.DataFrame({'revenue': ts.values[:6]}, index=idx[:6])
f['yesterday'] = f.revenue.shift(1)     # correct: looks back
f['tomorrow']  = f.revenue.shift(-1)    # LEAKAGE: looks forward
print(f.round(1))
print()
print('Only ever shift positively for model features.')


## 9. Finding the gaps you cannot see


In [ ]:
patchy = ts.drop(ts.index[[10,11,12,40]])
full = patchy.resample('D').size()
print(f'{(full == 0).sum()} days with no data')
print(full[full == 0].index.strftime('%Y-%m-%d').tolist())


Missing days are almost always meaningful. Find them before averaging over them.


## Try it yourself

1. Parse `'15-Mar-2026'` with an explicit format.
2. Localise a timestamp inside the autumn clock change and see what pandas says.
3. Build a 28-day rolling mean with `min_periods=14` and explain the trade.
